# 07 — Per-study and aggregate clinical/text metrics

This notebook computes nothing while the human labeler-validation gate is pending. A pending gate ends at a clear planned checkpoint rather than raising a Python exception. A passed gate must satisfy every prespecified threshold and match the exact frozen labeled-generation cohort before analysis begins.

After those checks, the notebook computes FER, abnormal-case FER, omission, label F1/accuracy, RadGraph F1, CIDEr, BERTScore F1, ROUGE-L, and measured runtime from the corrected records. Missing metric packages produce explicit missing values, never zeros.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
import pandas as pd, numpy as np
from rerun_code.common import read_jsonl, write_json, write_jsonl
from rerun_code.config import sha256_path
from rerun_code.metrics import compute_text_metrics, aggregate_label_metrics
from rerun_code.report_labeler import LABELS_13, _cohort_fingerprint

gate_path = PATHS["labeler"] / "validation" / "validation_gate.json"
instructions_path = PATHS["labeler"] / "validation" / "HUMAN_ANNOTATION_NEXT_STEPS.txt"
status_path = PATHS["metrics"] / "notebook07_status.json"
analysis_ready = False
gate = {}
if not gate_path.exists():
    pending_reason = "Notebook 06 has not created validation_gate.json."
else:
    gate = json.loads(gate_path.read_text(encoding="utf-8"))
    pending_reason = str(gate.get("pending") or "Human validation has not passed.")
    analysis_ready = bool(gate.get("passed"))

# A stale gate file can retain passed=true while omitting the metrics
# written by Notebook 06. Treat that state as pending validation rather
# than allowing Notebook 07 to fail with an opaque KeyError or assertion.
if analysis_ready:
    required_gate_fields = (
        "required_n", "required_macro_f1", "required_f1_evaluable_labels",
        "n", "n_f1_evaluable_labels", "macro_f1",
    )
    missing_gate_fields = [
        field for field in required_gate_fields
        if field not in gate or gate.get(field) is None
    ]
    if missing_gate_fields:
        analysis_ready = False
        pending_reason = (
            "validation_gate.json is marked passed=true but is incomplete "
            f"(missing values: {missing_gate_fields}). "
            "Rerun Notebook 06 cell 3; do not edit this file manually."
        )
    else:
        try:
            required_n = int(gate["required_n"])
            required_macro_f1 = float(gate["required_macro_f1"])
            required_labels = int(gate["required_f1_evaluable_labels"])
            gate_errors = []
            if int(gate["n"]) < required_n:
                gate_errors.append(f"validated n={gate.get('n')} is below {required_n}")
            if int(gate["n_f1_evaluable_labels"]) != required_labels:
                gate_errors.append(
                    f"evaluable labels={gate.get('n_f1_evaluable_labels')} rather than {required_labels}"
                )
            if float(gate["macro_f1"]) < required_macro_f1:
                gate_errors.append(
                    f"macro F1={gate.get('macro_f1')} is below {required_macro_f1}"
                )
        except (TypeError, ValueError, KeyError) as exc:
            gate_errors = [f"validation metrics are not numeric ({exc})"]
        if gate_errors:
            analysis_ready = False
            pending_reason = (
                "validation_gate.json is marked passed=true but does not satisfy the "
                "prespecified gate: " + "; ".join(gate_errors) + ". "
                "Rerun Notebook 06 cell 3 after correcting the annotation files."
            )

write_json(status_path, {
    "ready": False,
    "gate_path": str(gate_path),
    "gate_passed": bool(gate.get("passed")),
    "pending_reason": None if analysis_ready else pending_reason,
})

if not analysis_ready:
    print("HUMAN VALIDATION PENDING — NOTEBOOK 07 DID NOT COMPUTE METRICS.")
    print("Reason:", pending_reason)
    if instructions_path.exists():
        print("Instructions:", instructions_path)
    print("Next action: rerun Notebook 06 cell 3 to regenerate validation_gate.json, then rerun Notebook 07.")
else:
    labeled_path = PATHS["labels"] / "labeled_generation.jsonl"
    if not labeled_path.exists():
        raise FileNotFoundError(f"Notebook 06 labeled cohort is missing: {labeled_path}")
    frame = pd.DataFrame(read_jsonl(labeled_path))
    required_columns = {
        "generation_record_id", "final_report", "reference_report", "reference_vector",
        "prediction_vector", "model_key", "bundle", "source_dataset", "condition",
        "total_generation_seconds", "total_verifier_seconds",
    }
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise KeyError(f"Labeled generation cohort is missing columns: {missing_columns}")
    if frame.empty or frame["generation_record_id"].astype(str).duplicated().any():
        raise AssertionError("Labeled generation cohort is empty or has duplicate identifiers")
    if frame["final_report"].fillna("").astype(str).str.strip().eq("").any():
        raise AssertionError("Labeled generation cohort contains an empty report")
    for vector_column in ("reference_vector", "prediction_vector"):
        invalid_vectors = frame[vector_column].map(
            lambda value: not isinstance(value, list)
            or len(value) != len(LABELS_13)
            or any(item not in (0, 1) for item in value)
        )
        if invalid_vectors.any():
            raise AssertionError(
                f"{vector_column} contains {int(invalid_vectors.sum())} invalid vectors"
            )

    frozen_fingerprint = gate.get("annotation_manifest", {}).get(
        "generation_cohort_fingerprint"
    )
    actual_fingerprint = _cohort_fingerprint(frame)
    if not frozen_fingerprint or actual_fingerprint != frozen_fingerprint:
        raise AssertionError(
            "The labeled generation cohort does not match the cohort frozen for human validation"
        )

    frame["runtime_seconds"] = (
        frame["total_generation_seconds"].astype(float)
        + frame["total_verifier_seconds"].astype(float)
    )
    frame = compute_text_metrics(frame)
    per_study_path = PATHS["metrics"] / "per_study_metrics.jsonl"
    aggregate_path = PATHS["metrics"] / "aggregate_metrics.csv"
    text_metric_audit_path = PATHS["metrics"] / "text_metric_runtime_audit.json"
    write_json(text_metric_audit_path, {
        "radgraph_legacy_encode_plus_compatibility_attempted": True,
        "radgraph_f1_reward_component": "partial_entity_relation_RG_ER",
        "errors": dict(frame.attrs.get("text_metric_errors", {})),
    })
    write_jsonl(per_study_path, frame.to_dict("records"))
    rows = []
    grouping = ["model_key", "bundle", "source_dataset", "condition"]
    for keys, group in frame.groupby(grouping, dropna=False):
        result = aggregate_label_metrics(group)
        rows.append(dict(zip(grouping, keys), **result))
    aggregate = pd.DataFrame(rows)
    aggregate.to_csv(aggregate_path, index=False)
    write_json(status_path, {
        "ready": True,
        "gate_path": str(gate_path),
        "gate_passed": True,
        "generation_cohort_fingerprint": actual_fingerprint,
        "n_records": int(len(frame)),
        "per_study_metrics": str(per_study_path),
        "per_study_metrics_sha256": sha256_path(per_study_path),
        "aggregate_metrics": str(aggregate_path),
        "aggregate_metrics_sha256": sha256_path(aggregate_path),
        "text_metric_runtime_audit": str(text_metric_audit_path),
    })
    print("NOTEBOOK 07 COMPLETE — METRICS ARE LINKED TO THE PASSED VALIDATION COHORT.")
    display(aggregate.sort_values(["bundle", "model_key", "condition"]))